In [59]:
import pandas as pd 
import os 
from sqlalchemy import create_engine, String, Date, Float, inspect, text, MetaData
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker


In [60]:
zfspath = '/mnt/zfs-104' # change based on how zfs-104 is mounted
data_path = 'solar_financial_volatility/downloaded_csvs' # path to data 

In [61]:
user = 'cgirlamo' # change to your user - need to have .pgpass file in your home directory
h = '129.24.196.99' # keep the same 
p = '5432' # keep the same 
db = 'sv_db' # keep the same
db_url = f'postgresql+psycopg2://{user}@{h}:{p}/{db}' # keep the same
engine = create_engine(db_url) # keep the same 
session = sessionmaker(engine)
class Base(DeclarativeBase):
    pass

In [62]:
try:
    with engine.connect() as connection:
        result = connection.execute(text('SELECT version();'))
        print("Success")
except Exception as e:
    print(e)

Success


In [63]:
Base.metadata.reflect(bind=engine)
Base.metadata.drop_all(engine)
Base.metadata.clear()
path = os.path.join(zfspath,data_path)
for file in os.listdir(path):
    if file[-3:] == 'csv':
        nf = os.path.join(path,file)

        df = pd.read_csv(nf)
        # print(df.columns)
        df['Value'] = df.iloc[:,1]
        df['Date'] = df.iloc[:,0]
        try:
            df['Value'] = df['Value'].astype('float64')
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        except Exception as e:
            print(e)
        class_att = {
            '__tablename__': file[:-4],
            'Date': mapped_column(Date,primary_key=True),
            'Value': mapped_column(Float)
        }
        type(file[:-4], (Base,), class_att)
        df.to_sql(
            name=file[:-4],
            con=engine,
            if_exists='append',
            index=False
        )
Base.metadata.create_all(engine)

/tmp/ipykernel_1568296/3291880545.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
/tmp/ipykernel_1568296/3291880545.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


could not convert string to float: 'Bull-Bear'


/tmp/ipykernel_1568296/3291880545.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


could not convert string to float: 'na'


In [65]:
ins = inspect(engine)
tables = ins.get_table_names(schema='public')
# meta = MetaData()
# meta.reflect(bind=engine, schema='public')
# tables = list(meta.tables.values())
dfs = []
for name in tables:
    df = pd.read_sql_table(name,con= engine, columns=['Date', 'Value'])
    df = df.rename(columns={'Value': name}).set_index('Date')
    dfs.append(df)
m = pd.concat(dfs, join='outer').reset_index()
m['Date'] = pd.to_datetime(m['Date'], errors='coerce')
m.to_sql(name='full_dataset', con=engine, if_exists='replace',index=False)


177